# 4.6 Proteomics: Blood Proteins in Type 2 Diabetes

**Omics series, class 6 of 7.** DNA (4.1–4.4) is the instruction book; RNA (4.5) shows which instructions are being read; **proteins** are what the body actually builds and uses: hormones, enzymes, signals. *All of Us* has measured about **5,400 proteins** in blood from about **10,000 participants**, with a technology called **Olink**.

Our questions:

1. **Which blood proteins are higher or lower in people with type 2 diabetes?**
2. **Are those proteins controlled by genetic variants (pQTLs)?** And does rs7903146 affect any protein?

### By the end of this notebook you'll be able to

1. Explain how Olink measures proteins, and what an **NPX** value is
2. Find the *All of Us* proteomics files, and read only the rows you need from a very large file
3. Apply **QC** flags, and reshape long data into a protein × person matrix
4. Run a **differential abundance** analysis with `limma`, and read the results
5. Look up **pQTLs** (variants linked to protein levels) in the *All of Us* results

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | the *All of Us* proteomics files (v9) and the Controlled Tier dataset |
| 💾 **Saves** | `proteomics_t2d_results.csv`: one row per protein |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup and a tour of the proteomics files | 15 |
| 3. The cohort | 10 |
| 4. Load our people's proteins | 15 |
| 5. QC, reshape, explore | 20 |
| 6. Differential abundance | 25 |
| 7. pQTLs | 15 |
| Wrap-up | 5 |

**Before you start:**

- This notebook works **on its own**. It's an **R** notebook in `notebooks/omics/`, and loads `../aou_setup.R`.
- It uses **limma**, installed by the setup cell (as in 4.5; quick if you've run 4.5).

> 💰 **Cost:** runs on a **standard app**. The cohort query costs cents to under $1. Reading our people's rows from the 2.6 GB protein file takes **about 5 minutes** (the whole compressed file is read once) and costs pennies. Stop the app when you're done.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including any person's protein values), or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### From genes to proteins

| | DNA (4.1–4.4) | RNA (4.5) | Protein (today) |
|---|---|---|---|
| What it is | the instructions | a working copy of one gene | the product |
| Changes with disease? | no (you're born with it) | yes | yes |
| Measured in *All of Us* | genomes, arrays | RNA-seq, whole blood | **Olink**, blood plasma |
| Participants (v9) | about 535,000 | about 9,000 | about 10,000 |

Proteins in the blood come from many organs (liver, fat, muscle, the immune system, the pancreas), so they can reflect what's happening **throughout the body**, not only in blood cells. That's one reason proteins are promising **biomarkers**.

### How Olink measures proteins

Olink uses **pairs of antibodies**. Each antibody sticks to a different part of one protein, and each carries a short **DNA tag**. Only when **both** antibodies bind the same protein molecule are their tags close enough to join into a short DNA sequence, which is then counted by **DNA sequencing**. More protein → more joined tags → more counts.

- The *All of Us* platform is **Olink Explore HT**: about 5,400 proteins (**assays**) from one small blood sample.
- Each assay has an **OlinkID** (e.g. `OID20049`), the protein's **UniProt** ID, and a short name (**Assay**, usually the gene name, e.g. `LEP` for leptin).

### NPX: Olink's unit

Olink reports **NPX** (*Normalized Protein eXpression*):

- It's on a **log₂ scale**, like log-CPM in 4.5: a difference of **1** means **twice** as much protein.
- It's **relative**, not an absolute concentration: you can compare NPX for the **same protein** between people, but not between two different proteins.
- It's normalized to control samples on each **plate**, to remove differences between plates. The file we use has also been **batch-normalized** across all plates by *All of Us*.

### Differential abundance

The same idea as differential expression in 4.5: for **each protein**, compare cases with controls, adjusted for age and sex, and correct for testing ~5,400 proteins (**adjusted p-value**, FDR). We use **limma** again, but without `voom()`: NPX values are already on a log scale and aren't counts.

### pQTLs

A **pQTL** (*protein quantitative trait locus*) is a variant linked to a protein's level: the protein version of an eQTL (4.5). **Cis**-pQTLs are near the protein's own gene. *All of Us* has calculated them, for everyone combined (**COMB**) and per ancestry group.

### Glossary

| Term | Means |
|---|---|
| **Proteomics** | measuring many proteins at once |
| **Plasma** | the liquid part of blood, without the cells |
| **Assay** | one protein measurement on the Olink panel |
| **NPX** | Olink's relative, log₂-scale protein level |
| **Long format** | one row per person per protein (vs **wide**: one column per person) |
| **Differential abundance** | a difference in a protein's level between groups |
| **pQTL** | a variant linked to a protein's level |

## Part 2. Setup and a tour of the proteomics files

The setup cell loads `aou_setup.R` and, if needed, installs **limma** from Bioconductor (5–10 minutes the first time; quick if you ran 4.5).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

# limma, from Bioconductor (first time only)
user_lib <- Sys.getenv("R_LIBS_USER")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))
for (pkg in c("data.table", "BiocManager")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg, lib = user_lib, repos = "https://cloud.r-project.org")
}
if (!requireNamespace("limma", quietly = TRUE)) BiocManager::install("limma", lib = user_lib, update = FALSE, ask = FALSE)
suppressPackageStartupMessages(library(limma))

PROT_DIR <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
list.files(PROT_DIR)
list.files(file.path(PROT_DIR, "npx", "batch_normalized"))

**🔍 Read the output:**

| File or folder | What's in it |
|---|---|
| `manifest.tsv` | one row per **sample**: its participant (`research_id`), plate, and file locations |
| `npx/batch_normalized/` | **all** samples in one file, batch-normalized across plates. `…_reps_removed` has **one sample per person** (repeat measurements removed): **what we use** |
| `npx/tsv/`, `npx/parquet/` | the original files, one per **batch** of plates (`LCSET-…`), before batch normalization |
| `olink_report/` | Olink's PDF quality report for each batch |
| `pqtl/` | precomputed **pQTL** results (Part 7) |

### The manifest

Column names and counts only: the rows are participant IDs.

In [ ]:
manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

names(manifest)
nrow(manifest)                     # samples
n_distinct(manifest$research_id)   # participants

**🔍 Read the output:** there are a few more **samples** than **participants**: some people were measured twice (**replicates**, to check the measurements are repeatable). The `_reps_removed` file keeps one sample per person.

### The protein file

`Olink_10k_aou_v9_reps_removed.tsv.gz` is in **long format**: one row per person **per protein**. That's about 10,000 people × 5,400 proteins: **over 50 million rows**, 2.6 GB compressed. Far too big to load whole.

We read just the **first line** (the column names) with a command-line pipe: `gzip -dc` uncompresses, `head -1` stops after one line.

In [ ]:
NPX_FILE <- file.path(PROT_DIR, "npx", "batch_normalized", "Olink_10k_aou_v9_reps_removed.tsv.gz")

npx_columns <- str_split(system(str_glue("gzip -dc {shQuote(NPX_FILE)} | head -1"), intern = TRUE), "\t")[[1]]
npx_columns

**🔍 Read the output:** 36 columns. The ones we need:

| Column | Means |
|---|---|
| `ResearchID` | the participant (same as `person_id`) |
| `SampleID` | the sample |
| `OlinkID`, `UniProt`, `Assay` | the protein: Olink's ID, the UniProt ID, and its short name (usually the gene) |
| `NPX` | the protein level (log₂ scale) |
| `SampleQC` | `PASS` or `FAIL`: did this sample's measurement in this block of the panel pass Olink's checks? |
| `AssayQC` | `PASS` or not: did the protein's assay pass on this plate? |

Others describe the plate (`PlateID`, `WellID`), the panel's **blocks** (Explore HT is run in blocks of assays), alternative NPX versions (`ExtNPX`, `PCNormalizedNPX`), and software versions.

## Part 3. The cohort

The same type 2 diabetes cohort as in 4.1–4.5, kept to people with proteomics, aged 18 to 100, sex at birth *Female* or *Male*. Then 300 cases and 300 controls at random.

💰 The query costs cents to under $1.

In [ ]:
cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

# Everyone with proteomics, by cohort group
cohort %>% filter(person_id %in% manifest$research_id) %>% count(cohort) %>% hide_small_counts()

prot_cohort <- cohort %>%
  filter(person_id %in% manifest$research_id,
         cohort %in% c("case", "control"),
         between(age, 18, 100),
         sex_at_birth %in% c("Female", "Male")) %>%
  mutate(group = factor(cohort, levels = c("control", "case")))

N_PER_GROUP <- 300
set.seed(2026)
samples <- prot_cohort %>%
  arrange(person_id) %>%
  group_by(group) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

samples %>%
  group_by(group) %>%
  summarise(n = n(), mean_age = round(mean(age), 1), pct_female = round(100 * mean(sex_at_birth == "Female"), 1))

**🔍 Read the output:** how many people with proteomics are cases, controls, or excluded; then our 300 + 300. As in 4.5, cases are usually **older**, so age goes in the model.

## Part 4. Load our people's proteins

We use **awk**, a small command-line tool for text tables, to keep only **our 600 people's rows** and **the columns we need**, before R sees anything:

| Part of the command | What it does |
|---|---|
| `gzip -dc file` | uncompresses the file as it's read |
| `awk … 'NR==FNR {keep[$1]; next}'` | first reads `ids.txt` (our 600 IDs) into a list called `keep` |
| `FNR==1 \|\| ($col in keep)` | then keeps the header line, and every line whose `ResearchID` is in `keep` |
| `{print $a, $b, …}` | prints only the columns we need |

`match()` finds each column's **position** in the header, because awk refers to columns by number (`$27`), not name.

`fread(cmd = …)` (from **data.table**, as in 4.5) runs the command and reads the result.

⏱️ About **5 minutes**: the whole 2.6 GB file is uncompressed and scanned once. A good moment for questions.

In [ ]:
ids_file <- file.path(tempdir(), "ids.txt")
write_lines(samples$person_id, ids_file)

wanted   <- c("ResearchID", "SampleID", "OlinkID", "UniProt", "Assay", "NPX", "SampleQC", "AssayQC")
col_nums <- match(wanted, npx_columns)
stopifnot(!anyNA(col_nums))   # stops if a column name has changed

awk_command <- str_glue(
  "gzip -dc {shQuote(NPX_FILE)} | ",
  "awk -F'\\t' -v OFS='\\t' -v id={col_nums[1]} ",
  "'NR==FNR {{keep[$1]; next}} FNR==1 || ($id in keep) {{print {str_c('$', col_nums, collapse = ', ')}}}' ",
  "{shQuote(ids_file)} -"
)

start <- Sys.time()
npx_long <- data.table::fread(cmd = awk_command, sep = "\t", header = TRUE,
                              colClasses = c(ResearchID = "character", NPX = "numeric")) %>%
  as_tibble()
Sys.time() - start

dim(npx_long)
n_distinct(npx_long$ResearchID)   # people found
n_distinct(npx_long$OlinkID)      # proteins

**🔍 Read the output:** about 600 × 5,400 = 3 million rows, our 600 people (if a few are missing, their samples weren't in this file), and about 5,400 proteins.

The doubled braces `{{ }}` in the command are how `str_glue()` writes a literal `{`: single braces are for inserting R values.

## Part 5. QC, reshape, explore

### QC flags

Measurements that **failed** QC shouldn't be used. We count them, then set their NPX to missing (`NA`).

In [ ]:
npx_long %>% count(SampleQC, AssayQC)   # rows (person x protein), not people

# People with at least one failed block (don't share if 1 to 20)
npx_long %>% filter(SampleQC == "FAIL") %>% summarise(people = n_distinct(ResearchID)) %>%
  rename(n = people) %>% hide_small_counts()

npx_long <- npx_long %>%
  mutate(NPX = if_else(SampleQC == "PASS" & AssayQC == "PASS", NPX, NA_real_))

**🔍 Read the output:** most rows **pass**. A sample can fail in one **block** of the panel and pass in the others, so we drop failed measurements, not whole people.

### Long to wide

limma wants a **matrix**: one row per **protein**, one column per **person** (like the count matrix in 4.5). `pivot_wider()` turns the long table into that shape (see **2.2 reshape**). We name rows by `OlinkID`, which is unique; `Assay` names can repeat.

In [ ]:
protein_info <- npx_long %>% distinct(OlinkID, UniProt, Assay)

npx_wide <- npx_long %>%
  select(OlinkID, ResearchID, NPX) %>%
  pivot_wider(names_from = ResearchID, values_from = NPX, values_fn = mean)

npx <- as.matrix(npx_wide[, -1])
rownames(npx) <- npx_wide$OlinkID

# Put the sample table in the same order as the columns
samples <- samples %>% filter(person_id %in% colnames(npx))
npx     <- npx[, samples$person_id]
stopifnot(identical(colnames(npx), samples$person_id))

dim(npx)   # proteins x people

# Share of missing values per protein
missing_share <- rowMeans(is.na(npx))
summary(missing_share)

**🔍 Read the output:** about 5,400 proteins × 600 people. Most proteins have few or no missing values (from failed QC).

We keep proteins measured in at least **90%** of our people. limma can handle the remaining gaps.

In [ ]:
npx <- npx[missing_share <= 0.10, ]
dim(npx)

# NPX for one person: roughly centred on 0, on a log2 scale
hist(npx[, 1], breaks = 50, main = "NPX values, one person", xlab = "NPX (log2 scale)")

### PCA

As in 4.5: the big patterns. PCA can't handle missing values, so **for the PCA only** we fill each gap with that protein's average (a simple **imputation**). The differential abundance test in Part 6 uses the real values, gaps and all.

In [ ]:
filled <- npx
gaps   <- which(is.na(filled), arr.ind = TRUE)                    # row and column of each gap
filled[gaps] <- rowMeans(npx, na.rm = TRUE)[gaps[, "row"]]       # the protein's average

pca <- prcomp(t(filled), scale. = TRUE)
pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)

pca_df <- tibble(PC1 = pca$x[, 1], PC2 = pca$x[, 2], group = samples$group,
                 sex_at_birth = samples$sex_at_birth, age = samples$age)

ggplot(pca_df, aes(PC1, PC2, colour = group)) +
  geom_point(alpha = 0.6) +
  labs(x = str_glue("PC1 ({pct[1]}%)"), y = str_glue("PC2 ({pct[2]}%)"), colour = NULL,
       title = "PCA of blood proteins")

**🔍 Read the output:** do cases and controls separate? As with RNA, usually only a little: diabetes changes some proteins, but it isn't the biggest source of variation. `scale. = TRUE` puts every protein on the same scale first, so no single protein dominates.

### ✍️ Your turn

Colour the PCA by **age** instead (`colour = age`). Does PC1 or PC2 follow age?

<details>
<summary><b>Show answer</b></summary>

```r
ggplot(pca_df, aes(PC1, PC2, colour = age)) +
  geom_point(alpha = 0.6)
```

Many blood proteins change with **age** (e.g. *GDF15*), so age often lines up with a main PC. That's one more reason to adjust for it.

</details>

In [ ]:
# Your code here

## Part 6. Differential abundance

For every protein: **NPX ~ group + age + sex at birth**, testing `groupcase`. NPX is already on a log₂ scale, so we use `lmFit()` directly (no `voom()`, which is for counts).

In [ ]:
design <- model.matrix(~ group + age + sex_at_birth, data = samples)

fit     <- eBayes(lmFit(npx, design))
results <- topTable(fit, coef = "groupcase", number = Inf) %>%
  rownames_to_column("OlinkID") %>%
  left_join(protein_info, by = "OlinkID") %>%
  as_tibble()

results %>%
  select(Assay, OlinkID, logFC, AveExpr, P.Value, adj.P.Val) %>%
  head(20)

results %>%
  filter(adj.P.Val < 0.05) %>%
  count(direction = if_else(logFC > 0, "higher with diabetes", "lower with diabetes"))

**🔍 Read the output:**

| Column | Means |
|---|---|
| `Assay` | the protein's short name |
| `logFC` | the difference in NPX, **case − control**: 1 = twice as much in people with diabetes |
| `AveExpr` | the protein's average NPX |
| `adj.P.Val` | the **adjusted p-value** (FDR) across all proteins. **Use this one.** |

Proteins tend to show **more** differences than blood RNA: they come from organs all over the body. Look for proteins linked to **insulin resistance**, **fat tissue**, **the liver**, **the kidneys**, and **inflammation**.

### Volcano plot

In [ ]:
ggplot(results, aes(logFC, -log10(P.Value), colour = adj.P.Val < 0.05)) +
  geom_point(size = 0.8, alpha = 0.6) +
  geom_text(data = head(results, 10), aes(label = Assay), vjust = -0.6, size = 3,
            check_overlap = TRUE, show.legend = FALSE) +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "#08519c"), name = "Adjusted p < 0.05") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  theme_bw() +
  labs(x = "logFC (NPX, type 2 diabetes vs control)", y = "-log10(p-value)",
       title = "Blood proteins in type 2 diabetes")

**🔍 Read the output:** proteins high and far to the right are clearly **higher** in diabetes; far to the left, **lower**. The 10 most significant are labelled.

### The top protein

In [ ]:
top <- results$OlinkID[1]

tibble(NPX = npx[top, ], group = samples$group) %>%
  filter(!is.na(NPX)) %>%
  ggplot(aes(group, NPX)) +
  geom_boxplot() +
  theme_bw() +
  labs(title = results$Assay[1], y = "NPX (log2 scale)", x = NULL)

In [ ]:
save_data(results, "proteomics_t2d_results.csv")

### ✍️ Your turn

1. **Leptin** (`LEP`) is made by **fat tissue**: more body fat, more leptin. Is it higher in people with diabetes? What might explain that, and what variable would you want to add to the model?
2. Look up ***TCF7L2***: is it on the Olink panel at all?

<details>
<summary><b>Show answer</b></summary>

```r
results %>% filter(Assay %in% c("LEP", "TCF7L2")) %>% select(Assay, logFC, adj.P.Val)
```

1. Leptin is usually **higher** in people with type 2 diabetes, largely because they have more body fat on average. That's **confounding by body size**: to separate diabetes from obesity, you'd add **BMI** (from the physical measurements in the Controlled Tier dataset) to the model.
2. If no row comes back, *TCF7L2* isn't measured. It's a **transcription factor**: it works inside the cell's nucleus and isn't released into the blood, so a blood protein panel isn't designed to measure it. Another reminder that each omics layer shows only part of the picture.

</details>

In [ ]:
# Your code here

## Part 7. pQTLs

*All of Us* has tested each protein against nearby variants (**cis-pQTLs**), in the same format as the eQTL results in 4.5: one row per protein, with its **strongest** nearby variant. These are summary statistics: no individual data.

First, what does `phenotype_id` (the protein) look like: an OlinkID, a UniProt ID, or a gene name? We check which one matches.

In [ ]:
pqtl <- read_tsv(file.path(PROT_DIR, "pqtl", "cis_pqtl", "AoU_v9_COMB.cis_pqtl.txt.gz"), show_col_types = FALSE)

nrow(pqtl)   # proteins tested

c(OlinkID = mean(pqtl$phenotype_id %in% protein_info$OlinkID),
  UniProt = mean(pqtl$phenotype_id %in% protein_info$UniProt),
  Assay   = mean(pqtl$phenotype_id %in% protein_info$Assay))

**🔍 Read the output:** the share of `phenotype_id` values that match each kind of protein ID. The one close to **1** is how this file names proteins. The next cell uses whichever matches best.

### Do our top proteins have pQTLs?

In [ ]:
id_shares <- c(OlinkID = mean(pqtl$phenotype_id %in% protein_info$OlinkID),
               UniProt = mean(pqtl$phenotype_id %in% protein_info$UniProt),
               Assay   = mean(pqtl$phenotype_id %in% protein_info$Assay))
pqtl_id <- names(which.max(id_shares))
pqtl_id

results %>%
  filter(adj.P.Val < 0.05) %>%
  head(20) %>%
  inner_join(pqtl, by = setNames("phenotype_id", pqtl_id)) %>%
  select(Assay, logFC, adj.P.Val, variant_id, af, slope, qval) %>%
  arrange(qval)

**🔍 Read the output:** for our top diabetes-related proteins, each one's **strongest** nearby variant. A small `qval` means the protein has a clear **pQTL**: its level is partly set by **genetics**.

**💬 Discuss:** if a protein is higher in diabetes **and** has a strong pQTL, could you use the pQTL to ask whether the protein **causes** diabetes? (This is the idea behind **Mendelian randomization**: genetic variants are fixed at birth, so if variants that raise a protein also raise diabetes risk in a GWAS, the protein may be on the causal path. It needs careful checks, but it's how pQTLs are often used.)

### Does rs7903146 affect any protein?

The **fine-mapped** pQTL file lists, for each protein, the variants most likely to be the **causal** pQTL (**credible sets**), with gene names. We look for rs7903146 by position.

In [ ]:
finemapped <- data.table::fread(
  file.path(PROT_DIR, "pqtl", "finemapped_pqtl", "AoU_COMB_pQTL_SusieMerged.annotated.tsv"),
  select = c("chromosome", "position", "variant", "molecular_trait_id", "gene_name", "pip", "posterior_mean")
) %>% as_tibble()

nrow(finemapped)

finemapped %>%
  filter(position == 112998590, str_detect(as.character(chromosome), "10"))

**🔍 Read the output:** each row is a protein for which rs7903146 is in a **credible set**. `pip` (*posterior inclusion probability*) is the chance that this variant is the causal one, from 0 to 1; `posterior_mean` is its estimated effect.

If the table is **empty**, rs7903146 isn't a likely cause of any measured protein's level. That fits 4.5: rs7903146 has little effect on blood RNA, and its main effects are thought to be in the **pancreas**, on insulin secretion.

## ⚠️ Common errors

| Message | Means |
|---|---|
| *stopifnot(!anyNA(col_nums)) is not TRUE* | A column name in the protein file has changed. Look at `npx_columns` and edit `wanted` |
| `fread` returns 0 rows | The IDs in `ids.txt` didn't match `ResearchID`. Check they're text, not numbers like `1e+06` |
| The awk step takes much longer than 5 minutes | The app may be small or busy. It has to read the whole 2.6 GB file once |
| *there is no package called 'limma'* | Rerun the setup cell and wait for the install to finish |
| `inner_join` returns no rows in Part 7 | None of the top proteins matched `phenotype_id`. Look at `head(pqtl$phenotype_id)` |

## ✅ Key takeaways

- **Olink** measures thousands of proteins with antibody pairs and DNA tags; **NPX** is a relative, log₂-scale level
- Big **long-format** files can be filtered on the command line (**awk**) before R reads them
- Use **QC flags**; reshape long → wide for matrix-based tools like limma
- Blood proteins come from the **whole body**, so they often show more disease differences than blood RNA
- **pQTLs** link genetics to protein levels, and can help ask whether a protein is **causal**

**Next:** **4.7 rnaseq_proteomics**: for people with **both** RNA and protein data, do a gene's RNA and its protein go up and down together?

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).